In [26]:
import pandas as pd

clusters = pd.read_csv(
    "../constants/Astroviridae_Aves_Amphibia_14072026.csv",
    sep=","
)

clusters["ID"] = clusters["ID"].str.replace("NC-", "NC_", regex=False)

annotated = pd.read_csv(
    "../constants/Astroviridae_30062026_annotated_MAstV_species.tsv",
    sep="\t",
    low_memory=False
)

host = annotated[["Accession", "Host_species"]].drop_duplicates("Accession")

result = clusters.merge(
    host,
    left_on="ID",
    right_on="Accession",
    how="left"
).drop(columns=["Accession"])

print("Всего строк:", len(result))
print("С host_species:", result["Host_species"].notna().sum())
print("Без host_species:", result["Host_species"].isna().sum())

print("\nID без host_species:")
print(
    result.loc[
        result["Host_species"].isna(),
        "ID"
    ].to_string(index=False)
)

output = "../constants/Astroviridae_Aves_Amphibia_14072026_host.csv"

result.to_csv(
    output,
    index=False
)

print("\nРезультат сохранён:")
print(output)

Всего строк: 407
С host_species: 352
Без host_species: 55

ID без host_species:
 PV793829
 PV793828
 MW588064
 MW588063
 MW588062
 OQ986679
 OQ986609
 OQ986608
 OQ986607
 OQ986677
 OQ986606
 MW826483
 MW826482
 MW385412
 MW826371
 MT138011
 MT138010
 MT138007
 MT138005
 MT138002
 MT137999
 MT137996
 MT138009
 MT137995
 MT137994
 MT137992
 MT137991
 MN920672
 MN920671
 MN920670
 MN920669
 MN920668
 MN920667
 MN920663
 MK096775
 MK096774
 MK096773
 MG661261
NC_005790
 KF753807
 KF753806
 KF753805
 KF753804
 EU143851
 EU143850
 EU143849
 EU143848
 EU143847
 EU143846
 EU143845
 EU143844
 EU143843
 AF206663
 HV511037
 AB033998

Результат сохранён:
../constants/Astroviridae_Aves_Amphibia_14072026_host.csv


In [15]:
import pandas as pd

clusters = pd.read_csv(
    "../constants/Astroviridae_Aves_Amphibia_14072026.csv",
    sep=","
)

clusters["ID"] = clusters["ID"].str.replace("NC-", "NC_", regex=False)

ids = [
    "MW588063",
    "MW588062",
    "OQ986679",
    "OQ986609",
    "OQ986608",
    "OQ986607",
    "OQ986677",
    "OQ986606",
    "MT138011",
    "MT138010",
    "MT138007",
    "MT138005",
    "MT138002",
    "MT137999",
    "MT137996",
    "MT138009",
    "MT137995",
    "MT137994",
    "MT137992",
    "MT137991",
    "MN920672",
    "MN920671",
    "MN920670",
    "MN920669",
    "MN920668",
    "MN920667",
    "MN920663",
    "MG661261",
    "NC_005790",
    "KF753807",
    "KF753806",
    "KF753805",
    "KF753804",
    "EU143851",
    "EU143850",
    "EU143849",
    "EU143848",
    "EU143847",
    "EU143846",
    "EU143845",
    "EU143844",
    "EU143843",
    "AF206663",
    "HV511037",
    "AB033998"
]

result = clusters.loc[
    clusters["ID"].isin(ids),
    ["ID", "Host"]
].copy()

print("Запрошено ID:", len(ids))
print("Найдено ID:", result["ID"].nunique())
print("Не найдено ID:", len(set(ids) - set(result["ID"])))

print("\nID без Host:")
print(
    result.loc[
        result["Host"].isna(),
        "ID"
    ].to_string(index=False)
)

missing_ids = sorted(set(ids) - set(result["ID"]))

if missing_ids:
    print("\nID, которых нет в исходной таблице:")
    print("\n".join(missing_ids))

input_taxonomy = result[["Host"]].drop_duplicates()

output = "../constants/hosts_for_taxonomy.tsv"

input_taxonomy.to_csv(
    output,
    sep="\t",
    index=False
)

print("\nФайл для taxonomy-скрипта сохранён:")
print(output)

Запрошено ID: 45
Найдено ID: 45
Не найдено ID: 0

ID без Host:
 MT138011
 MT138010
 MT138007
 MT138005
 MT138002
 MT137996
 MT138009
 MT137995
 MT137994
 MT137992
 MN920672
 MN920671
 MN920670
 MN920669
 MN920668
 MN920663
NC_005790
 HV511037

Файл для taxonomy-скрипта сохранён:
../constants/hosts_for_taxonomy.tsv


In [27]:
import pandas as pd

host = pd.read_csv(
    "../constants/Astroviridae_Aves_Amphibia_14072026_host.csv",
    sep=",",
    dtype=str
)

taxonomy = pd.read_csv(
    "../constants/hosts_taxonomy.tsv",
    sep="\t",
    dtype=str
)

taxonomy = taxonomy[
    ["host_original", "species"]
].drop_duplicates("host_original")

host = host.merge(
    taxonomy,
    how="left",
    left_on="Host",
    right_on="host_original"
)

if "Host_species" not in host.columns:
    host["Host_species"] = pd.NA

mask = (
    host["Host_species"].isna() |
    (host["Host_species"].str.strip() == "")
)

host.loc[mask, "Host_species"] = host.loc[mask, "species"]

host = host.drop(
    columns=["host_original", "species"]
)

host.to_csv(
    "../constants/Astroviridae_Aves_Amphibia_14072026_host_species.csv",
    index=False
)